In [ ]:
import os
import torch
import numpy as np
import pickle
import umap.umap_ as umap
from tabular_model import *
from tabular_util import *

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True

_, config = load_config_yaml("config.yaml")
config["device"] = torch.device("cpu")  # ('cuda:'+ config['gpu'])


In [ ]:
def infer(model, infer_csv_path, output_name, umap_model=None):
    model.eval()

    infer_df = pd.read_csv(infer_csv_path)
    infer_dataset = LongitudinalSingleDataset(infer_df)
    inferDataLoader = DataLoader(
        infer_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=0,
    )

    result_dir = f'./results/{config["dataset_name"]}/{config["model_name"]}/{ckpt_label}/'
    os.makedirs(result_dir, exist_ok=True)
    path = os.path.join(result_dir, output_name if output_name.endswith(".npy") else output_name + ".npy")

    rid_list, tab_list, lb_list, recon_list, z_list, age_list = [], [], [], [], [], []

    with torch.no_grad():
        for _, sample in enumerate(inferDataLoader, 0):
            tab = sample["tab"].to(config["device"], dtype=torch.float).unsqueeze(1)
            if not torch.isfinite(tab).all():
                nan_indices = torch.nonzero(torch.isnan(tab))
                for i, _, j in nan_indices:
                    tab[i, _, j] = torch.nanmean(tab[:, :, j])
            zero_mx = torch.zeros_like(tab)
            zs, recons = model(tab, zero_mx)

            rid_list.extend(sample["rid"])
            tab_list.append(tab.detach().cpu().numpy())
            recon_list.append(recons[0].detach().cpu().numpy())
            z_list.append(zs[0].detach().cpu().numpy())
            age_list.append(sample["age"].numpy())
            lb_list.append(sample["lb"].detach().cpu().numpy())

    # Concatenate arrays
    tab_list = np.concatenate(tab_list, axis=0)
    recon_list = np.concatenate(recon_list, axis=0)
    z_list = np.concatenate(z_list, axis=0)
    age_list = np.concatenate(age_list, axis=0)
    lb_list = np.concatenate(lb_list, axis=0)

    # UMAP processing
    if umap_model is None:
        umap_model = umap.UMAP(n_neighbors=15, n_components=4, random_state=seed)
        umap_embedding = umap_model.fit_transform(z_list)
        pickle_path = os.path.join(result_dir, "umap_transformer.pkl")
        with open(pickle_path, "wb") as f:
            pickle.dump(umap_model, f)
        print(f"Fitted new UMAP and saved transformer to {pickle_path}")
    else:
        umap_embedding = umap_model.transform(z_list)

    # Save results
    results = {
        "RID": np.array(rid_list),
        "age": age_list,
        "lb": lb_list,
        "tab": tab_list,
        "recon": recon_list,
        "z": z_list,
        "umap_embedding": umap_embedding,
    }
    # np.save(path, results, allow_pickle=True)
    # print(f"Saved inference results to {path}")
    return results


def load_ckpt(ckpt_path):
    flag, config_load = load_config_yaml(os.path.join(ckpt_path, "config.yaml"))
    # load config file
    if flag:
        print("load yaml config file")
        for key in config_load.keys():  # if yaml has, use yaml's param, else use config
            if key == "phase" or key == "gpu" or key == "continue_train" or key == "ckpt_name":
                continue
            if key in config.keys():
                config[key] = config_load[key]
            else:
                print("current config do not have yaml param")
    else:
        save_config_yaml(ckpt_path, config)

    # Load model
    model = LSP(
        num_neighbours=config["num_neighbours"],
        dims=config["dims"],
        agg_method=config["agg_method"],
        gpu=config["device"],
        activation="leakyrelu",
        dropout=0.0,
        slope=0.2,
        batch_norm=True,
    ).to(config["device"])

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=1e-5, amsgrad=True)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=5, min_lr=1e-5)

    [optimizer, scheduler, model], start_epoch = load_checkpoint_by_key(
        [optimizer, scheduler, model],
        ckpt_path,
        ["optimizer", "scheduler", "model"],
        config["device"],
        config["ckpt_name"],
    )
    print(model.parameters())

    return config, model, optimizer, scheduler, start_epoch


######################
# Inference settings #
######################

trial = 2
infer_name = "train"
ckpt_dir = "./ckpt/ADNI1GO234/LSP/"
ckpt_label = next(folder for folder in os.listdir(ckpt_dir) if folder.startswith(f"trial{trial}_"))
config["ckpt_path"] = os.path.join("./ckpt/", config["dataset_name"], config["model_name"], ckpt_label)

config, model, optimizer, scheduler, start_epoch = load_ckpt(config["ckpt_path"])

infer_csv_path = f"data/ADNI1GO234/splits/trial{trial}/preadj_{infer_name}.csv"

output_name = f"{infer_name}_results.npy"
# umap_path = os.path.join("./results/", config["dataset_name"], config["model_name"], ckpt_label, "umap_transformer.pkl")
print(f"Inference for {infer_name} data, trial {trial}...")
results = infer(model, infer_csv_path, output_name, umap_model=None)


In [ ]:
umap_path = "results/ADNI1GO234/LSP/trial2_2025_9_24_15_24/umap_transformer.pkl"
with open(umap_path, "rb") as f:
    umap_model = pickle.load(f)

print("Loaded UMAP parameters:", umap_model.get_params())

if hasattr(umap_model, "embedding_"):
    embedding = umap_model.embedding_
    print("UMAP embedding shape:", embedding.shape)
else:
    raise ValueError("Loaded UMAP model does not contain precomputed embeddings.")

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.scatter(embedding[:, 0], embedding[:, 1], s=10, alpha=0.7)
plt.title("UMAP Embedding Visualization")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
import pandas as pd
import plotly.express as px

umap_data = results["umap_embedding"]
legend = results["lb"].flatten()

if umap_data.shape[1] < 3:
    raise ValueError("UMAP embedding must have at least 3 dimensions for 3D visualization.")

df = pd.DataFrame(
    umap_data[:, :3],
    columns=["UMAP1", "UMAP2", "UMAP3"],
)
df["Legend"] = legend

# Highlight only the target value, others grey
target_value = 4   # Change the highlighted value here.

df["Color"] = df["Legend"].apply(lambda x: x if x == target_value else "Other")

fig = px.scatter_3d(
    df,
    x="UMAP1", y="UMAP2", z="UMAP3",
    color="Color",
    color_discrete_map={
        target_value: "red",
        "Other": "lightgray",
    },
    opacity=0.8,
    title=f"UMAP 3D Embedding Highlight (Legend = {target_value})",
    hover_data=["Legend", "UMAP1", "UMAP2", "UMAP3"],
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(
    width=700,
    height=600,
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()


In [ ]:
import plotly.express as px

umap_data = results["umap_embedding"]
legend = results["lb"].flatten()

dim_count = min(4, umap_data.shape[1])
columns = [f"UMAP{i+1}" for i in range(dim_count)]

df = pd.DataFrame(
    umap_data[:, :dim_count],
    columns=columns,
)
df["Legend"] = legend

fig = px.scatter_matrix(
    df,
    dimensions=columns,
    color="Legend",
    title="Scatter Matrix of UMAP Embedding Colored by Legend",
    opacity=0.7,
    hover_data=["Legend"],
)
fig.update_traces(diagonal_visible=False)
fig.update_layout(
    width=800,
    height=800,
    margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()
